# PharmaML — The Complete ML Workflow (Zero to One Hundred)

One notebook, every stage of a QSAR machine-learning project, in the order
you would actually run it:

| § | Stage | Toolkit |
|---|---|---|
| 1 | **Data entry** — import *your own* CSV/Excel | `pharmaml.data` |
| 2 | **Feature calculation** — SMILES → descriptors, fingerprints, similarity | `pharmaml.chem` |
| 3 | Exploratory analysis — audit, class balance, descriptor ranges | `pharmaml.viz` |
| 4 | Endpoint transformation — IC50 → pIC50 | `pharmaml.chem` |
| 5 | Split — stratified train/test | `pharmaml.ml` |
| 6 | Train — grid-searched random forest, CV folds | `pharmaml.ml` |
| 7 | Evaluate — confusion matrix, ROC, metrics table | `pharmaml.viz` |
| 8 | Multiclass recipe — K-class target, K selectable | `pharmaml.ml` |
| 9 | Regression — parity + residual diagnostics | `pharmaml.viz` |
| 10 | Leaderboard — every model, cross-validated | `pharmaml.ml` |
| 11 | Registry tour — all 24 families, one table | `pharmaml.ml` |
| 12 | Interpret — feature importance | `pharmaml.viz` |
| 13 | Deploy — save/load, score new compounds | `pharmaml.ml` |
| 14 | **Trust** — applicability domain & prediction reliability | `pharmaml.ml` |
| 15 | Publish — one-click HTML report + png/svg/pdf + csv/md/json | `pharmaml.viz` |


**Setup** — install the package with the plotting extra first:

```bash
pip install -e ".[reports]"
```

Every demo table below ships in `examples/data/` (regenerated deterministically
by `scratch/build_example_data.py`), so the notebook runs offline and stays
fully reproducible; replace the demo file in §1 with your own and re-run all
cells.


> **No-output variant.** This notebook is the bandwidth-friendly twin of
> `complete_ml_workflow.ipynb`: identical cells, every output and inline figure stripped
> (5% of the original size). Open the full twin to read the
> pre-computed results; run this one top-to-bottom (`Shift+Enter`) to
> produce them yourself — the setup cell lists the optional dependencies
> each stage needs.


In [ ]:
import sys
import tempfile
from pathlib import Path

import pandas as pd

from pharmaml import __version__

pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)

# Scratch directory for this run's artifacts (use any path you like).
WORK = Path(tempfile.mkdtemp(prefix="ml_workflow_"))

# The shipped demo inventory (examples/data) -- this notebook loads
# its demo tables from there instead of fabricating them inline.
DATA_DIR = Path("data")

print(f"pharmaml {__version__}")
print(f"artifacts -> {WORK}")


## 1 · Data entry — bring your own file

The legacy scripts hardcoded dataset filenames inside ~30 `.py` files.
`load_dataset` is the single entry point for **your** data: point `MY_FILE`
at any CSV/TSV/TXT/Excel file (delimiter sniffed automatically), and the
target column is auto-detected, the task inferred (classification with any
number of classes vs regression), non-numeric columns dropped *visibly*.

CLI equivalents:

```bash
pharmaml data inspect --data my_compounds.csv
pharmaml ml train    --data my_compounds.csv --task auto
```

The demo file carries a `SMILES` column that `load_dataset` drops as
non-numeric — §2 turns those structures into features.


In [ ]:
# --- DEMO DATA ---------------------------------------------------------
# A realistic QSAR table shipped with the toolkit: 8 descriptors, a
# 3-level activity target, plus SMILES/ID metadata columns that should
# NOT become features.  (In your project: set MY_FILE to your own path.)
MY_FILE = DATA_DIR / "qsar_classification.csv"   # <- replace with your own file

frame_demo = pd.read_csv(MY_FILE)
print(f"demo table: {MY_FILE}  ({len(frame_demo)} rows x {frame_demo.shape[1]} columns)")


In [ ]:
# --- THE ACTUAL IMPORT (works unchanged for your file) -----------------
from pharmaml.data import load_dataset

ds = load_dataset(MY_FILE)          # target auto-detected -> 'activity'

print(ds.summary())
print("\nfeature columns:", ds.feature_names)
print("dropped (non-numeric):", ds.dropped_columns)

## 2 · Feature calculation — from SMILES to descriptors

The legacy archive scattered feature computation over four scripts
(`Descriptor calculator.py`, `Morgan_calculator.py`, `ECFP_calculation.py`,
`Mordred_calculation.py`) that silently assumed *"we are sure all SMILES are
valid"*. `pharmaml.chem` unifies them with explicit accounting:

* `compute_descriptors(smiles)` — all 2D RDKit descriptors (~217 columns),
  a Morgan count table, or the 1800+ Mordred suite (`descriptor_set=...`);
* `compute_fingerprints(smiles)` — Morgan/ECFP, RDKit path, atom-pair
  and Avalon fingerprints, binary bits or counts (Avalon and Morgan
  both have a count flavour — the archive's `Descriptor
  classification.py` feature unions folded 512-wide Avalon counts);
* `compute_maccs(smiles)` — the 167-bit MACCS keys;
* `tanimoto_matrix(bits, bits)` — pairwise similarity for heat-maps;
* invalid SMILES are dropped *visibly* (row indices logged), never silently.

Requires RDKit — `pip install 'pharmaml[chem]'`.


In [ ]:
from pharmaml.chem import compute_descriptors, compute_fingerprints, compute_maccs

# Silence RDKit's C++ parser spam -- pharmaml logs invalid rows explicitly.
try:
    from rdkit import RDLogger
    RDLogger.DisableLog("rdApp.*")
except ImportError:
    pass

# A demo screening library of real drug-like structures (77 valid
# molecules), plus two *deliberately* broken entries so the invalid-SMILES
# accounting is visible.  (In your project: use your own SMILES column --
# frame["SMILES"] works directly.)
library_table = pd.read_csv(DATA_DIR / "screening_library.csv")
LIBRARY = list(library_table.itertuples(index=False, name=None))  # (name, smiles) pairs

smiles_series = pd.Series([smi for _, smi in LIBRARY], name="SMILES")

descriptor_frame = compute_descriptors(smiles_series, progress=False)

print(f"{descriptor_frame.shape[0]} valid molecules x "
      f"{descriptor_frame.shape[1] - 1} descriptors (RDKit 2D set)")
print(f"dropped {len(LIBRARY) - len(descriptor_frame)} invalid SMILES ")


In [ ]:
# A chemist-readable slice of the 217 computed columns.
KEY_DESCRIPTORS = ["MolWt", "MolLogP", "TPSA", "NumHDonors", "NumHAcceptors",
                   "NumRotatableBonds", "LabuteASA", "RingCount"]

key_frame = descriptor_frame[["SMILES"] + KEY_DESCRIPTORS].round(2)
key_frame.index = [LIBRARY[i][0] for i in descriptor_frame.index]
key_frame.index.name = "compound"
key_frame

In [ ]:
# Fingerprints: binary bits vs counts, four different families.
bits_morgan, valid_morgan = compute_fingerprints(smiles_series, kind="morgan", n_bits=2048)
counts_morgan, _ = compute_fingerprints(smiles_series, kind="morgan", n_bits=2048,
                                        as_counts=True)
bits_path, _ = compute_fingerprints(smiles_series, kind="rdkit", n_bits=1024)
bits_pair, _ = compute_fingerprints(smiles_series, kind="atom_pair", n_bits=2048)
bits_avalon, _ = compute_fingerprints(smiles_series, kind="avalon", n_bits=512)
counts_avalon, _ = compute_fingerprints(smiles_series, kind="avalon", n_bits=512,
                                        as_counts=True)
maccs, valid_maccs = compute_maccs(smiles_series)

print(f"morgan bits   : {bits_morgan.shape}  (mean on-bit rate {bits_morgan.mean():.3f})")
print(f"morgan counts : {counts_morgan.shape}")
print(f"rdkit path    : {bits_path.shape}")
print(f"atom pair     : {bits_pair.shape}")
print(f"avalon bits   : {bits_avalon.shape}  -- the rar Descriptor wrappers folded 512-wide Avalon counts")
print(f"avalon counts : {counts_avalon.shape}  (mean nonzero counts {counts_avalon.astype(bool).mean(axis=1).mean():.3f})")
print(f"MACCS keys    : {maccs.shape}  (valid rows: {len(valid_maccs)})")


In [ ]:
from pharmaml.viz import plot_descriptor_distributions

fig_desc = plot_descriptor_distributions(
    descriptor_frame[KEY_DESCRIPTORS],
    title="Computed descriptor distributions (library)",
)

In [ ]:
from pharmaml.chem import tanimoto_matrix
from pharmaml.viz import plot_similarity_heatmap

similarity = tanimoto_matrix(bits_morgan, bits_morgan)
labels = [LIBRARY[i][0] for i in valid_morgan]

fig_similarity = plot_similarity_heatmap(
    similarity, labels=labels,
    title="Library similarity — Tanimoto on Morgan bits",
)

In [ ]:
# Structures in, model out: the computed descriptors become X.  The
# shipped table pairs them with an activity column (LogP + H-bond donors
# + noise, generated once from the library descriptors) -- the role your
# measured IC50/pIC50 column plays in a real project.
struct_file = DATA_DIR / "library_with_descriptors.csv"   # <- replace with your own file

ds_struct = load_dataset(struct_file)   # target auto-detected -> 'activity'
print(ds_struct.summary())


In [ ]:
from pharmaml.ml import make_train_test_split, train_model, evaluate_model

sX_tr, sX_te, sy_tr, sy_te = make_train_test_split(
    ds_struct.X, ds_struct.y, test_size=0.25, random_state=42
)
struct_result = train_model("logistic_regression", "classification", sX_tr, sy_tr,
                            cv=3, tune=True)
struct_report = evaluate_model(struct_result.model, sX_te, sy_te, "classification")

print(f"{struct_result.name}: CV ROC-AUC = {struct_result.cv_mean:.3f} "
      f"± {struct_result.cv_std:.3f}  ({len(ds_struct.y)}-compound demo library)")
print(f"held-out accuracy: {struct_report.accuracy:.2f} on {len(sy_te)} compounds")

In [ ]:
from pharmaml.viz import plot_feature_importance

fig_struct_importance = plot_feature_importance(
    struct_result.model, feature_names=ds_struct.feature_names, top_n=8,
    title="Which computed descriptors drive activity?",
)

## 3 · Exploratory data analysis

Look at the data *before* modelling: the audit summary, the class balance
(one bar per class, counts annotated) and the descriptor distributions
(histogram grid with means marked) surface problems the legacy scripts
discovered as cryptic sklearn crashes.


In [ ]:
from pharmaml.viz import plot_class_distribution, plot_descriptor_distributions

fig_balance = plot_class_distribution(ds, title="Class distribution (activity)")

In [ ]:
fig_dists = plot_descriptor_distributions(
    ds.X, columns=ds.feature_names[:6], title="Descriptor distributions (first 6)"
)

In [ ]:
# Per-class quick statistics for one descriptor.
ds.frame.groupby("activity")["descr_00"].describe().round(3)

## 4 · Endpoint transformation — IC50 → pIC50

Potency data arrives as IC50 (nM); models regress on the log-transformed
pIC50. `add_pic50_column` appends the converted column to any table.


In [ ]:
from pharmaml.chem import add_pic50_column

potency = pd.DataFrame(
    {"compound": ["A", "B", "C", "D"],
     "IC50 (nM)": [12.0, 240.0, 1850.0, 92.0]}
)
converted = add_pic50_column(potency, source_col="IC50 (nM)")
converted.round(3)

## 5 · Train / test split

`make_train_test_split` stratifies by the target so every class appears in
both partitions with the same proportions.


In [ ]:
from pharmaml.ml import make_train_test_split

X_train, X_test, y_train, y_test = make_train_test_split(
    ds.X, ds.y, test_size=0.2, random_state=42
)

split_audit = pd.DataFrame([
    {"partition": "train", "rows": len(y_train),
     **{f"n_{cls}": int((y_train == cls).sum()) for cls in ds.classes}},
    {"partition": "test", "rows": len(y_test),
     **{f"n_{cls}": int((y_test == cls).sum()) for cls in ds.classes}},
])
split_audit

## 6 · Train one model (with grid search)

`train_model` wraps the estimator in the impute+scale pipeline and runs a
5-fold cross-validated grid search; per-fold scores are returned so the
mean is never mistaken for a guarantee.


In [ ]:
from pharmaml.ml import train_model

result = train_model(
    "random_forest", "classification", X_train, y_train, cv=5, tune=True
)
print(f"{result.name}: CV ROC-AUC = {result.cv_mean:.3f} ± {result.cv_std:.3f} "
      f"({result.fit_seconds:.1f}s)")
print("best params:", result.best_params)

In [ ]:
from pharmaml.viz import plot_cv_scores

fig_cv = plot_cv_scores(result, metric="ROC-AUC (ovo)")

## 7 · Recipe A — evaluate the two-class (binary) classifier

The `ClassificationReport` carries every metric plus the confusion matrix;
the plots below are the two figures the legacy scripts produced —
confusion heat-map and ROC curves — with count/share annotations. With
exactly two classes the report reads like a classic active/inactive
screen: a single ROC-AUC and a 2×2 matrix. (Recipe B — any number of
classes — is §8.)


In [ ]:
from pharmaml.ml import evaluate_model

report = evaluate_model(result.model, X_test, y_test, "classification")
print(report)

metrics_frame = (
    pd.DataFrame([report.as_dict()]).T.rename(columns={0: "value"}).round(4)
)
metrics_frame

In [ ]:
from pharmaml.viz import plot_confusion_matrix

fig_confusion = plot_confusion_matrix(report, title="Confusion matrix (held-out test)")

In [ ]:
# Row-normalized view: per-class recall shares along the diagonal.
fig_confusion_norm = plot_confusion_matrix(
    report, normalize=True, title="Confusion matrix — row-normalized"
)

In [ ]:
from pharmaml.viz import plot_roc_curves

fig_roc = plot_roc_curves(result.model, X_test, y_test, title="ROC — one vs rest")

## 8 · Multiclass — two or more classes

A target with 3+ distinct values is a first-class citizen: stratified
splits, k×k confusion matrix, one-vs-rest ROC per class, macro-averaged
metrics.


In [ ]:
# --- Recipe B: K classes — you pick K ------------------------------------
# The 4-class demo table ships as examples/data/qsar_multiclass.csv,
# built by examples/make_synthetic_qsar_data.py with seed 11 -- the same
# generator you can run for any other class count K.
N_CLASSES = 4                                   # <-- select: 2, 3, 4, 5, ...
ACTIVITY_LABELS = ["inactive", "weak", "moderate", "strong", "potent"]

multi_file = DATA_DIR / "qsar_multiclass.csv"   # <- replace with your own file
ds_multi = load_dataset(multi_file)
print(ds_multi.summary())


In [ ]:
mX_tr, mX_te, my_tr, my_te = make_train_test_split(ds_multi.X, ds_multi.y, test_size=0.25)

mc = train_model("logistic_regression", "classification", mX_tr, my_tr, tune=False)
mc_report = evaluate_model(mc.model, mX_te, my_te, "classification")

print(f"{N_CLASSES}-class accuracy={mc_report.accuracy:.3f}  f1_macro={mc_report.f1_macro:.3f}  "
      f"roc_auc(ovr)={mc_report.roc_auc:.3f}")

fig_multi_confusion = plot_confusion_matrix(mc_report, title=f"Multiclass confusion ({N_CLASSES} classes)")

In [ ]:
fig_multi_roc = plot_roc_curves(mc.model, mX_te, my_te, title=f"Multiclass ROC — {N_CLASSES} classes, one vs rest")

## 9 · Regression path

The same pipeline with a continuous pIC50 target: parity scatter against
the identity line, and residual diagnostics (vs predicted + histogram)
that the legacy scripts never drew.  And when the assay contains gross
outliers, ``huber_regression`` keeps the fit honest where plain squared
error bends.


In [ ]:
# The regression demo table ships as examples/data/qsar_regression.csv
# (make_synthetic_qsar_data.py, seed 21): 20 descriptors + a continuous
# pIC50 endpoint.
reg_file = DATA_DIR / "qsar_regression.csv"     # <- replace with your own file
ds_reg = load_dataset(reg_file)                 # numeric target -> regression

print(ds_reg.summary())
print("task:", ds_reg.task, "| target:", ds_reg.target)


In [ ]:
rX_tr, rX_te, ry_tr, ry_te = make_train_test_split(ds_reg.X, ds_reg.y, test_size=0.25)

reg_result = train_model("random_forest", "regression", rX_tr, ry_tr, tune=False)
reg_report = evaluate_model(
    reg_result.model, rX_te, ry_te, "regression", keep_predictions=True
)
print(reg_report)

In [ ]:
from pharmaml.viz import plot_predicted_vs_actual, plot_residuals

fig_parity = plot_predicted_vs_actual(
    ry_te, reg_report.predictions,
    metrics={"RMSE": reg_report.rmse, "R²": reg_report.r2},
    title="Regression — predicted vs actual (pIC50)",
)

In [ ]:
fig_residuals = plot_residuals(ry_te, reg_report.predictions, title="Regression residuals")

In [ ]:
from pharmaml.ml import train_model, evaluate_model

# Robust regression — huber_regression (upgrades Regression/Huber.py).
# Gross activity outliers wreck least-squares fits because squared error
# grows quadratically with the residual; Huber's linear-epsilon loss
# saturates instead. Corrupt 15 training pIC50 values by +8 log units --
# the full width of the assay-plausible window [3, 11], so every corrupted
# label lands beyond the ceiling (a mis-entered plate value) -- then judge
# both models on the *clean* held-out set:
ry_corrupt = ry_tr.copy()
ry_corrupt.iloc[:15] += 8.0

lin_corrupt = train_model("linear_regression", "regression", rX_tr, ry_corrupt, tune=False)
hub_corrupt = train_model("huber_regression", "regression", rX_tr, ry_corrupt, tune=False)
lin_clean = evaluate_model(lin_corrupt.model, rX_te, ry_te, "regression")
hub_clean = evaluate_model(hub_corrupt.model, rX_te, ry_te, "regression")

print(f"clean-test RMSE — linear_regression: {lin_clean.rmse:.3f}")
print(f"clean-test RMSE — huber_regression : {hub_clean.rmse:.3f}   (same corrupted training data)")
assert hub_clean.rmse < lin_clean.rmse, "Huber should beat plain least squares under outliers"

## 10 · Model leaderboard

`compare_models` cross-validates the whole registry at once — the job the
original archive spread over ~30 near-duplicate scripts. The error bars
are fold-to-fold standard deviations.


In [ ]:
from pharmaml.ml import compare_models

leaderboard = compare_models(
    ds.X, ds.y, "classification",
    models=["logistic_regression", "knn", "random_forest", "decision_tree"],
    cv=5,
)
leaderboard.round(4)

In [ ]:
from pharmaml.viz import plot_model_comparison

fig_leaderboard = plot_model_comparison(leaderboard, title="Model leaderboard (5-fold CV)")

## 11 · The full registry — every family, one tour

The leaderboard above compared four hand-picked families. The registry
holds **more — 24 model families**, each registered for one or both
tasks, with the optional engines (`xgboost`, `lightgbm`, `catboost` and
the `torch` deep nets) appearing automatically once their extra is
installed. The tour below fits **every** available family on the two
splits this notebook already built — the §5 classification split and
the §9 regression split — and scores each on the same held-out test
rows, so the numbers stay directly comparable:

- `available` — `yes`, or the missing requirement (an unavailable family
  is listed honestly, it does not vanish silently);
- `held-out` — accuracy (classification) / R² (regression) on the
  untouched test split;
- `fit_s` — the `train_model` wall-clock fit time;
- `cum fit_s` — the running fit budget in tour order, closing with one
  `(total)` row — the number a k-fold CV sweep multiplies by k, and
  Hyperband's successive halving (the tuning notebook's ladder) exists
  to cut.

`list_models()` is the self-updating guarantee: a family added to the
registry tomorrow joins this tour without any edit here. The same call
powers the desktop playground's model picker and the CLI's
`pharmaml ml models`, so the three surfaces cannot drift apart.


In [ ]:
from pharmaml.exceptions import DataValidationError
from pharmaml.ml.models import list_models
from pharmaml.ml import evaluate_model

# One tour, every family: list_models() walks the registry itself, so
# this cell never needs editing when a family joins (extra_trees at
# 1.2.41, gradient_boosting at 1.2.42, pls at 1.2.43 ...).
tour_rows = []
for task, X_tr, X_te, y_tr, y_te in (
    ("classification", X_train, X_test, y_train, y_test),
    ("regression", rX_tr, rX_te, ry_tr, ry_te),
):
    for entry in list_models(task, only_available=False):
        name = entry["name"]
        if not entry["available"]:
            tour_rows.append(
                {"family": name, "task": task,
                 "available": f"no ({entry['requires']} not installed)"}
            )
            continue
        try:
            tour_result = train_model(name, task, X_tr, y_tr, tune=False)
            tour_report = evaluate_model(tour_result.model, X_te, y_te, task)
            score = tour_report.accuracy if task == "classification" else tour_report.r2
            tour_rows.append(
                {"family": name, "task": task, "available": "yes",
                 "held-out": f"{score:.3f}", "fit_s": round(tour_result.fit_seconds, 2)}
            )
        except DataValidationError as refusal:
            # (1.2.93) count-contract families (multinomial_nb) refuse
            # real-valued descriptors with the typed error -- the tour
            # records the refusal as a row instead of dying, exactly
            # like the "not installed" rows above.
            tour_rows.append(
                {"family": name, "task": task,
                 "available": f"no (data contract: {str(refusal).split(' -- ')[0].split(': ', 2)[-1]})"}
            )

registry_tour = pd.DataFrame(tour_rows)
n_ok = int((registry_tour["available"] == "yes").sum())

# The cumulative-budget read: the running fit cost in tour order (what
# the sweep has spent once it reaches each family) plus one total row —
# the number a k-fold CV sweep multiplies by k, and successive halving
# (the tuning notebook's Hyperband ladder) exists to cut.
registry_tour["cum fit_s"] = (
    registry_tour["fit_s"].fillna(0.0).cumsum().round(2)
)
total_s = float(registry_tour["fit_s"].sum())
registry_tour.loc[len(registry_tour)] = {
    "family": "(total)", "task": "—",
    "available": f"{n_ok} fitted", "held-out": "—",
    "fit_s": round(total_s, 2), "cum fit_s": round(total_s, 2),
}
print(f"{len(registry_tour) - 1} registry entries — {n_ok} available in this "
      f"environment, each fitted on the same splits and scored on the "
      f"same held-out rows")
print(f"tour fit budget: {total_s:.1f} s total — a 5-fold CV sweep of the "
      f"same families would spend ~{5 * total_s:.0f} s; the tuning "
      f"notebook's Hyperband ladder is the machinery that cuts such "
      f"multi-config budgets by halving")
registry_tour


### Meta-ensembles — stack them, vote on them

The registry tour above trains one family at a time; the two meta-
ensembles instead *combine* families.  ``build_voting`` (the
``Voting.py`` twins) lets the members cast soft votes (probability
averaging, weightable) or hard majority votes; ``build_stacking``
(the ``Stacking.py`` twins) has the members predict out-of-fold and a
meta-model blend those predictions.  Both take registry names, so any
combination the registry knows is one list away.

In [ ]:
from pharmaml.ml import build_stacking, build_voting

# Voting (upgrades Classification/Voting.py): three families cast soft
# votes, the random forest's counts double.
vote_clf = build_voting(
    "classification",
    ["random_forest", "knn", "logistic_regression"],
    voting="soft",
    weights=[2, 1, 1],
)
vote_clf.fit(X_train, y_train)
vote_report = evaluate_model(vote_clf, X_test, y_test, "classification")
print(f"soft voting  · accuracy {vote_report.accuracy:.3f}  "
      f"ROC-AUC {vote_report.roc_auc:.3f}")

# Stacking (upgrades Classification/Stacking.py): out-of-fold level-0
# predictions, logistic blender on top.
stack_clf = build_stacking(
    "classification",
    ["random_forest", "knn", "logistic_regression"],
    meta_model="logistic_regression",
)
stack_clf.fit(X_train, y_train)
stack_report = evaluate_model(stack_clf, X_test, y_test, "classification")
print(f"stacking     · accuracy {stack_report.accuracy:.3f}  "
      f"ROC-AUC {stack_report.roc_auc:.3f}")

In [ ]:
# The regression twins (Voting reg.py / Stacking reg.py): members average
# predictions; the default ridge blender sits on the out-of-fold
# predictions.  train_model("voting"/"stacking", ...) reaches the same
# builders with the CV/tuning pipeline wrapped around them.
vote_reg = build_voting("regression", ["random_forest", "knn"])
vote_reg.fit(rX_tr, ry_tr)
print(f"voting regressor  · R² {vote_reg.score(rX_te, ry_te):.3f}")

stack_reg = build_stacking("regression", ["random_forest", "knn"])
stack_reg.fit(rX_tr, ry_tr)
print(f"stacking regressor · R² {stack_reg.score(rX_te, ry_te):.3f}")

## 12 · Interpret — feature importance

Which descriptors drive the model? Works for tree ensembles
(`feature_importances_`), linear models (`|coef_|`) and explicit mappings.


In [ ]:
from pharmaml.viz import plot_feature_importance

fig_importance = plot_feature_importance(
    result.model, feature_names=ds.feature_names, top_n=8,
    title="Random-forest feature importance",
)

## 13 · Deploy — persistence and scoring new compounds

`save_model` writes a joblib bundle *with provenance metadata*; `load_model`
round-trips it. New, unseen rows are scored with one `predict` call — and
`validate_bundle_input` audits any new frame against the bundle's recorded
contract *before* the first predict, naming every way the data has drifted.


In [ ]:
from pharmaml.ml import save_model, load_model

MODEL_PATH = WORK / "rf_activity.joblib"
save_model(
    result.model, MODEL_PATH,
    name=result.name, task="classification",
    features=ds.feature_names, best_params=result.best_params,
    metrics=report.as_dict(),
)
loaded, metadata = load_model(MODEL_PATH)
print("reloaded:", metadata["name"], "| trained:", metadata["trained_at"])

In [ ]:
# Score five compounds the model has never seen.
new_rows = ds.X.sample(5, random_state=99)
predictions = loaded.predict(new_rows)

scored = pd.DataFrame({
    "compound_id": ds.frame.loc[new_rows.index, "compound_id"].values,
    "predicted_activity": predictions,
})
scored.to_csv(WORK / "scored_new_compounds.csv", index=False)
scored

### Audit the input before you trust the prediction

The production failure mode is quiet: the model was saved months ago,
the CSV got regenerated, a column was renamed, a new one appeared — and
instead of a prediction you get a cryptic sklearn feature-count crash
deep inside `predict`. `validate_bundle_input` checks any new frame
against the bundle's recorded contract *before* the first call and
names every difference — the inference-time twin of the provenance
`save_model` wrote.


In [ ]:
from pharmaml.ml import validate_bundle_input

# The frame the model was actually trained on: the contract holds.
print(validate_bundle_input(MODEL_PATH, ds.X))
print()

# The months-later CSV: one feature dropped, one renamed, one brand-new
# column -- the three ways real data drifts, all named in one verdict.
drifted = ds.X.drop(columns=[ds.feature_names[0]]).rename(
    columns={ds.feature_names[1]: "renamed_feature"}
)
drifted["batch_id"] = 1.0

audit_drift = validate_bundle_input(MODEL_PATH, drifted)
print(audit_drift)
print("ok:", audit_drift.ok)


In [ ]:
# The subtlest drift: the same columns in a different order.  sklearn
# rejects the frame even though the *set* of names matches -- the audit
# catches it before predict() does.
reordered = ds.X[[ds.feature_names[1], ds.feature_names[0]] + ds.feature_names[2:]]
print(validate_bundle_input(MODEL_PATH, reordered))


The same audit ships from the terminal — `pharmaml ml check` prints the
bundle's provenance block plus the verdict and **exits 1 on failure**
(CI-friendly); `--target` drops the training label when the CSV still
carries it:

```text
pharmaml ml check --model rf_activity.joblib --data new.csv --target activity
```


## 14 · Trust — applicability domain & prediction reliability

A prediction nobody can judge is a rumor. The uploaded
`Applicability_domain_analysis.py` workflow asks the two questions that
turn it into evidence: **is the query compound inside the training
descriptor space** (mean kNN distance vs a percentile threshold of the
training set's own distances), and **how much does the answer wobble**
when the training set is bootstrap-resampled (the ensemble's per-row
spread vs its own training-side threshold)? Both answers fold into one
`reliable` verdict per compound.

The same analysis ships from the terminal:

```text
pharmaml ml ad --data my_compounds.csv --model ridge --out Applicability_Domain_Results.csv
pharmaml ml ad --bundle ridge.joblib --train-data train.csv --test-data test.csv
```


In [ ]:
from pharmaml.ml import applicability_domain

# The query set: the held-out rows PLUS one wild extrapolation -- a
# descriptor profile 30 standard deviations off the training cloud.
far_row = ds_reg.X.max() + 30.0 * ds_reg.X.std()
X_query = pd.concat(
    [rX_te, pd.DataFrame([far_row], columns=ds_reg.X.columns)],
    ignore_index=True,
)

ad = applicability_domain(reg_result.model, rX_tr, ry_tr, X_query, k=5, n_models=15)
print(f"distance threshold {ad.distance_threshold:.3g} -- "
      f"{int(ad.inside_ad.sum())}/{len(ad.inside_ad)} query rows inside the domain")
print(f"{ad.n_models}-member bootstrap ensemble in {ad.fit_seconds:.1f} s "
      f"({ad.uncertainty_kind}); uncertainty threshold "
      f"{ad.uncertainty_threshold:.3g}")
print(f"reliable verdicts: {int(ad.reliable.sum())}/{len(ad.reliable)} "
      "compound(s) -- inside AD AND low uncertainty")

names = [f"held-out #{i}" for i in range(len(rX_te))] + ["extrapolation"]
reliability = ad.to_frame(names=names)
reliability.tail(6)


In [ ]:
from pharmaml.viz import plot_applicability_domain

# The upload's combined figure: (a) the training distance density with
# the threshold, (b) every query compound at its kNN distance against
# the ensemble prediction, green inside / red outside the domain.
fig_ad = plot_applicability_domain(
    ad, y_label="predicted pIC50",
    title="Applicability domain — random forest (regression)",
)


## 15 · Publish — the one-click visual bundle

`save_visual_bundle` renders every figure and table into a directory **and**
a self-contained `report.html` (figures embedded as base64 — no external
assets, renders identical offline). Tables are exported as csv *and*
markdown; figures as png *and* svg *and* pdf.


In [ ]:
from pharmaml.viz import export_table, metrics_to_dataframe, save_visual_bundle

BUNDLE = WORK / "reports"
artifacts = save_visual_bundle(
    figures={
        "class_distribution": fig_balance,
        "descriptor_distributions": fig_desc,
        "similarity_heatmap": fig_similarity,
        "cv_scores": fig_cv,
        "confusion_matrix": fig_confusion,
        "roc_curves": fig_roc,
        "parity_plot": fig_parity,
        "residuals": fig_residuals,
        "model_leaderboard": fig_leaderboard,
        "feature_importance": fig_importance,
    },
    tables={
        "metrics": metrics_to_dataframe(report),
        "leaderboard": leaderboard,
        "computed_descriptors": key_frame,
        "split_audit": split_audit,
        "scored_compounds": scored,
    },
    metrics=report,
    out_dir=BUNDLE,
    title="QSAR activity model — complete workflow",
    subtitle=f"{result.name} · {ds.n_samples} compounds · {len(ds.classes)} classes",
    formats=("png", "svg"),
    close=True,    # figures already rendered inline above; free the memory
    pdf_report=True,   # + report.pdf: the legacy scripts' fpdf output, upgraded
)
pd.DataFrame({"artifact": [p.name for p in artifacts],
              "kind": [p.suffix.lstrip('.') for p in artifacts]})

In [ ]:
# The same tables in every format a report could need.
exports = [
    export_table(leaderboard, BUNDLE / "leaderboard.md"),
    export_table(metrics_to_dataframe(report), BUNDLE / "metrics.json"),
    export_table(split_audit, BUNDLE / "split_audit.html"),
    export_table(scored, BUNDLE / "scored_new_compounds.tsv"),
]
[p.name for p in exports]

In [ ]:
# The self-contained report: open it in any browser — figures are embedded.
report_html = BUNDLE / "report.html"
size_kb = report_html.stat().st_size / 1024
print(f"{report_html}  ({size_kb:.0f} KB, zero external assets)")
print("First 3 lines:")
print("\n".join(report_html.read_text().splitlines()[:3]))

## 16 · The frozen feature set — replay a saved selection

`select_top_features` freezes a feature set as a small JSON ledger; `recalculate_selected_features` replays it on any new batch — only the families the selected columns need are computed, and the result carries exactly the recorded columns in the recorded rank order.  `selection_metadata.json` is the bundled top-24 of the descriptors + ecfp families over eight drug-like molecules, written by the factory's own saver — the same file the playground's **Recalculate from metadata** card loads.

In [ ]:
# Train once, select once, then recompute the SAME features for every
# new screening batch — no re-ranking, no vocabulary drift.
import json

from pharmaml.chem.features import recalculate_selected_features

frozen = json.loads(
    (DATA_DIR / "selection_metadata.json").read_text(encoding="utf-8")
)
fresh_batch = [
    "c1ccc(cc1)O",              # phenol
    "CCN(CC)C(=O)c1ccc(N)cc1",  # procainamide
    "NCCc1ccc(O)c(O)c1",        # dopamine
    "C1CCCCC1CO",               # cyclohexylmethanol
]
recalc_frame, recalc_report = recalculate_selected_features(
    fresh_batch, frozen
)
print(
    f"replayed {recalc_frame.shape[1]} selected features over "
    f"{recalc_report.n_valid} fresh molecules; "
    f"families recomputed: {recalc_report.families}"
)
recalc_frame.iloc[:, :6]

## 17 · Recap

| Stage | One-liner |
|---|---|
| Import your data | `ds = load_dataset("my.csv", target="activity")` |
| Compute descriptors | `compute_descriptors(frame["SMILES"])` |
| Compute fingerprints | `compute_fingerprints(smiles, kind="morgan")` |
| Similarity heat-map | `plot_similarity_heatmap(tanimoto_matrix(bits, bits))` |
| Audit | `ds.summary()` |
| Convert endpoints | `add_pic50_column(frame, "IC50 (nM)")` |
| Split | `make_train_test_split(ds.X, ds.y, 0.2)` |
| Binary or K classes | `make_classification_table(n_classes=N_CLASSES)` |
| Train | `train_model("random_forest", ds.task, Xtr, ytr)` |
| Evaluate | `evaluate_model(model, Xte, yte, ds.task)` |
| Compare | `compare_models(ds.X, ds.y, ds.task)` |
| Registry tour | `list_models(task)` — every family, fitted |
| Robust fit | `train_model("huber_regression", "regression", Xtr, ytr)` |
| Interpret | `plot_feature_importance(model, ds.feature_names)` |
| Deploy | `save_model(...)` / `load_model(path)` / `validate_bundle_input(path, X)` |
| Replay a frozen feature set | `recalculate_selected_features(new, metadata)` |
| Trust the prediction | `applicability_domain(model, Xtr, ytr, Xquery)` |
| Publish | `save_visual_bundle(figures, tables, metrics, "reports")` |
